# Exploratory Data Analysis — Student Study Strategy Dataset

**Course:** Machine Learning  
**Project:** Study Strategy Recommendation Agent  
**Author:** SleepQualityAgent / StudentAgent integration project  

---

## Objective

This notebook performs a structured exploratory data analysis (EDA) on the labelled dataset used to train a classifier that predicts `RecommendedStrategy` from student context features.

## Target variable

| Class | Meaning |
|-------|--------|
| `Rest` | Recovery priority — poor sleep / high fatigue |
| `BalancedStudy` | Steady, moderate study plan |
| `IntensiveStudy` | Focused push before an imminent exam |
| `LongTermPlan` | Early preparation with low urgency |

## Notebook structure

1. Dataset overview  
2. Missing value analysis  
3. Duplicate analysis  
4. Descriptive statistics  
5. Target distribution  
6. Numerical distributions  
7. Categorical distributions  
8. Correlation matrix  
9. Boxplots & outlier detection  
10. Pairplot  
11. Feature–target relationships  
12. IQR outlier report  
13. Summary report export  

> **Note:** Run all cells from the `ml/` directory. Figures are saved to `output/figures/`.

In [ ]:
from pathlib import Path
import sys

ML_ROOT = Path.cwd()
if ML_ROOT.name == "notebooks":
    ML_ROOT = ML_ROOT.parent
sys.path.insert(0, str(ML_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display, Markdown

from src.analysis.eda_helpers import (
    COLUMN_DESCRIPTIONS,
    STRATEGY_ORDER,
    build_markdown_report,
    column_overview_table,
    configure_plot_style,
    descriptive_statistics,
    detect_iqr_outliers,
    duplicate_summary,
    ensure_output_dirs,
    missing_value_summary,
    outlier_rows,
)
from src.analysis.eda_plots import (
    plot_boxplots_by_strategy,
    plot_categorical_counts,
    plot_correlation_matrix,
    plot_individual_boxplots,
    plot_missing_values,
    plot_numeric_histograms,
    plot_numeric_vs_strategy,
    plot_outlier_summary,
    plot_pairplot,
    plot_sleep_quality_vs_strategy,
    plot_target_distribution,
)
from src.data.schema import (
    CATEGORICAL_FEATURES,
    FEATURE_COLUMNS,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    load_dataset,
    load_schema,
)

configure_plot_style()
FIGURES_DIR, OUTPUT_DIR = ensure_output_dirs(ML_ROOT)
schema = load_schema()
df = load_dataset()

def show_figure(path: Path, width: int = 900) -> None:
    """Display a saved figure inline."""
    display(Image(filename=str(path), width=width))

print(f"Dataset loaded: {schema.source_path}")
print(f"Figures directory: {FIGURES_DIR}")

---
## 1. Dataset overview

We begin by inspecting the shape, column meanings, and data types. Understanding the schema ensures that subsequent analysis uses the correct feature roles (numeric vs categorical) and confirms alignment with the domain model in `Student.Domain`.

In [ ]:
n_rows, n_cols = df.shape
print(f"Shape: {n_rows} rows × {n_cols} columns")
print(f"Features: {len(FEATURE_COLUMNS)}  |  Target: 1 ({TARGET_COLUMN})")
print()
display(column_overview_table(FEATURE_COLUMNS, TARGET_COLUMN))
print()
display(df.dtypes.to_frame("DataType"))

In [ ]:
df.head(10)

**Observations:** The dataset contains 400 labelled records with 8 input features and one categorical target. Six features are numeric (`float` / `int`) and two are categorical strings (`SleepQuality`, `PreviousFeedback`). The first rows show plausible combinations — e.g. high stress with low sleep leading to `Rest`, or many days until exam with good sleep leading to `LongTermPlan`.

---
## 2. Missing value analysis

Missing data can bias models and reduce usable sample size. We report both counts and percentages, then visualise missingness per column.

In [ ]:
missing_df = missing_value_summary(df)
missing_df["MissingPercent"] = missing_df["MissingPercent"].astype(str) + "%"
display(missing_df)

missing_fig = plot_missing_values(df, FIGURES_DIR)
show_figure(missing_fig)

**Observations:** All columns report **0 missing values** (0%). The bar chart is flat at zero, confirming a complete dataset. This is expected for the current synthetic dataset and means imputation is not required at the preprocessing stage. In production, missing-value monitoring should remain part of the data validation pipeline.

---
## 3. Duplicate analysis

Duplicate rows inflate performance metrics and may indicate data collection errors. We check full-row duplicates and feature-only duplicates (same inputs, potentially different labels).

In [ ]:
dup_info = duplicate_summary(df)
print(f"Full duplicate rows:            {dup_info['duplicate_rows']}")
print(f"Duplicate feature combinations: {dup_info['duplicate_feature_rows']}")
print(f"Unique rows:                    {df.drop_duplicates().shape[0]} / {len(df)}")

**Observations:** No duplicate rows were found. Each record is unique, which supports reliable cross-validation and prevents optimistic bias from repeated samples.

---
## 4. Descriptive statistics

Summary statistics for all numerical features: mean, median, standard deviation, min, max, and quartiles (Q1, Q3).

In [ ]:
desc_stats = descriptive_statistics(df, NUMERIC_FEATURES)
display(desc_stats)

**Observations:**
- `DaysUntilExam` spans 0–30 days with high variance (std ≈ 8.6), reflecting both urgent and long-term scenarios.
- `StressLevel` and `FatigueLevel` are centred around the mid-range (mean ≈ 6.1 and 5.0).
- `SleepHours` averages 6.5 h with a relatively narrow spread (std ≈ 1.3).
- `HoursStudied` ranges from 0 to 10, showing diverse study intensities.

These ranges are consistent with realistic student behaviour and provide adequate variation for supervised learning.

---
## 5. Target variable distribution

Class balance affects metric choice and model fairness. We visualise the frequency of each recommended strategy.

In [ ]:
target_counts = df[TARGET_COLUMN].value_counts().reindex(STRATEGY_ORDER)
display(target_counts.to_frame("Count"))
display((target_counts / len(df) * 100).round(1).to_frame("Percent"))

target_fig = plot_target_distribution(df, TARGET_COLUMN, FIGURES_DIR)
show_figure(target_fig)

**Observations:** All four classes contain exactly **100 samples (25% each)**. The dataset is perfectly balanced, so accuracy is a meaningful metric alongside macro-F1. No resampling (SMOTE, undersampling) is needed for class imbalance.

---
## 6. Numerical feature distributions

Histograms with kernel density estimates reveal shape, skewness, and multimodality for each numeric feature.

In [ ]:
hist_fig = plot_numeric_histograms(df, NUMERIC_FEATURES, FIGURES_DIR)
show_figure(hist_fig, width=1000)

**Observations:**
- `DaysUntilExam` is bimodal — peaks at low values (urgent exams) and high values (long-term planning), matching the two temporal regimes in the data generation rules.
- `StressLevel` and `FatigueLevel` show spread across the 1–10 scale without extreme skew.
- `SleepHours` clusters around 5.5–7.5 h with a slight left tail (short sleep).
- `HoursStudied` is roughly uniform to mildly right-skewed.

No single feature appears constant or degenerate; all carry information for classification.

---
## 7. Categorical feature distributions

Count plots for `SleepQuality` and `PreviousFeedback` show category frequencies.

In [ ]:
for col in CATEGORICAL_FEATURES:
    print(f"\n{col}:")
    display(df[col].value_counts().to_frame("Count"))

cat_fig = plot_categorical_counts(df, CATEGORICAL_FEATURES, FIGURES_DIR)
show_figure(cat_fig, width=1000)

**Observations:** `SleepQuality` includes all three levels (Poor, Average, Good) with `Average` being most frequent — expected because moderate sleep is common in balanced and intensive profiles. `PreviousFeedback` is dominated by `None` and `Positive`, reflecting that many students have not yet provided feedback or found recommendations helpful. Both features will require one-hot or ordinal encoding during preprocessing.

---
## 8. Correlation matrix

Pearson correlation among numeric features helps identify redundancy and multicollinearity before modelling.

In [ ]:
corr = df[NUMERIC_FEATURES].corr().round(3)
display(corr)

corr_fig = plot_correlation_matrix(df, NUMERIC_FEATURES, FIGURES_DIR)
show_figure(corr_fig)

**Observations:**
- **Strong negative:** `StressLevel` ↔ `DaysUntilExam` (r ≈ −0.93) — urgency increases stress.
- **Strong negative:** `FatigueLevel` ↔ `DaysUntilExam` (r ≈ −0.83).
- **Strong positive:** `SleepHours` ↔ `DaysUntilExam` (r ≈ +0.82) — more time before exams correlates with better sleep.
- **Moderate negative:** `HoursStudied` ↔ `DaysUntilExam` — intensive cramming aligns with shorter horizons.

High correlations among predictors suggest that tree-based models may handle redundancy well, but linear models could benefit from regularisation or feature selection. An ablation study should test whether `SleepHours` adds value beyond `SleepQuality`.

---
## 9. Boxplots — outlier detection

Boxplots grouped by `RecommendedStrategy` highlight spread and potential outliers for each numeric feature.

In [ ]:
box_fig = plot_boxplots_by_strategy(df, NUMERIC_FEATURES, TARGET_COLUMN, FIGURES_DIR)
show_figure(box_fig, width=1000)

individual_box_paths = plot_individual_boxplots(df, NUMERIC_FEATURES, TARGET_COLUMN, FIGURES_DIR)
print(f"Saved {len(individual_box_paths)} individual boxplot figures.")

**Observations:**
- `Rest` consistently shows the highest `StressLevel` / `FatigueLevel` and the lowest `SleepHours`.
- `LongTermPlan` has the highest median `DaysUntilExam` and the lowest stress/fatigue medians.
- `IntensiveStudy` shows elevated `HoursStudied` with short `DaysUntilExam`.
- Individual points beyond whiskers represent statistical outliers (see Section 12 for IQR quantification).

Clear separation between class medians indicates that a classifier should achieve strong baseline performance.

---
## 10. Pairplot — numerical features by strategy

A pairplot visualises bivariate relationships while colouring points by the target class.

In [ ]:
pair_fig_path = plot_pairplot(df, NUMERIC_FEATURES, TARGET_COLUMN, FIGURES_DIR)
show_figure(pair_fig_path, width=1100)

**Observations:** The pairplot confirms that classes form distinguishable clusters in feature space. `DaysUntilExam` vs `StressLevel` shows the clearest class separation: `LongTermPlan` (green) occupies the high-days / low-stress corner, while `Rest` (blue) clusters at low days and high stress/fatigue. Some overlap exists between `BalancedStudy` and `IntensiveStudy` in the mid-range, which is where the model may need the most capacity.

---
## 11. Feature–target relationship plots

Focused visualisations for the features most relevant to study-strategy decisions.

In [ ]:
sleep_fig = plot_sleep_quality_vs_strategy(df, TARGET_COLUMN, FIGURES_DIR)
show_figure(sleep_fig)

stress_fig = plot_numeric_vs_strategy(df, "StressLevel", TARGET_COLUMN, FIGURES_DIR, "09_stress_level_vs_strategy.png")
show_figure(stress_fig)

fatigue_fig = plot_numeric_vs_strategy(df, "FatigueLevel", TARGET_COLUMN, FIGURES_DIR, "10_fatigue_level_vs_strategy.png")
show_figure(fatigue_fig)

days_fig = plot_numeric_vs_strategy(df, "DaysUntilExam", TARGET_COLUMN, FIGURES_DIR, "11_days_until_exam_vs_strategy.png")
show_figure(days_fig)

**Observations:**

| Plot | Key finding |
|------|-------------|
| **SleepQuality vs Strategy** | `Poor` sleep dominates `Rest`; `Good` sleep is most common in `LongTermPlan` and `BalancedStudy`. |
| **StressLevel vs Strategy** | `Rest` and `IntensiveStudy` show the highest stress; `LongTermPlan` the lowest. |
| **FatigueLevel vs Strategy** | `Rest` has the highest fatigue distribution; `LongTermPlan` the lowest. |
| **DaysUntilExam vs Strategy** | `LongTermPlan` is separated at 14+ days; `Rest` and `IntensiveStudy` cluster below 5 days. |

These four features are the strongest candidates for feature-importance analysis in the modelling phase.

---
## 12. IQR outlier detection

We flag outliers using the **Interquartile Range (IQR)** rule: a value is an outlier if it falls below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR`.

In [ ]:
outlier_summary = detect_iqr_outliers(df, NUMERIC_FEATURES)
display(outlier_summary)

outlier_sample = outlier_rows(df, NUMERIC_FEATURES)
print(f"\nRows flagged by at least one IQR rule: {len(outlier_sample)} ({len(outlier_sample)/len(df)*100:.1f}%)")
display(outlier_sample.head(10))

outlier_fig = plot_outlier_summary(outlier_summary, FIGURES_DIR)
show_figure(outlier_fig)

**Observations:** On this 400-row synthetic dataset, the IQR rule flags **0 rows** — values were generated within class-specific bounds. The bar chart confirms zero outliers per feature. The IQR procedure remains important for production data where genuine extremes or recording errors may appear. When outliers are detected, review them individually rather than removing automatically, because edge cases (e.g. exam today with maximum fatigue) are valid inputs the agent must handle.

---
## 13. Export summary report

A markdown report consolidating all findings is written to `output/eda_report.md` for inclusion in the project documentation.

In [ ]:
report_text = build_markdown_report(
    df=df,
    schema_path=schema.source_path,
    figures_dir=FIGURES_DIR,
    numeric_columns=NUMERIC_FEATURES,
    categorical_columns=CATEGORICAL_FEATURES,
    target_column=TARGET_COLUMN,
    missing_df=missing_value_summary(df),
    duplicate_info=dup_info,
    desc_stats=desc_stats,
    outlier_summary=outlier_summary,
    outlier_sample=outlier_sample,
)

report_path = OUTPUT_DIR / "eda_report.md"
report_path.write_text(report_text, encoding="utf-8")

saved_figures = sorted(p.name for p in FIGURES_DIR.glob("*.png"))
print(f"Report saved: {report_path}")
print(f"Figures saved: {len(saved_figures)} files in {FIGURES_DIR}")
for name in saved_figures:
    print(f"  - {name}")

---
## Conclusion

| Aspect | Finding |
|--------|--------|
| **Data quality** | Complete (no missing values), no duplicates |
| **Class balance** | Perfect — 100 samples per strategy |
| **Key discriminators** | `DaysUntilExam`, `StressLevel`, `FatigueLevel`, `SleepQuality` |
| **Correlations** | Strong coupling between urgency features and stress/fatigue |
| **Outliers** | Present but domain-valid; retain for training |
| **Next step** | Preprocessing pipeline → stratified split → baseline classifier |

All figures have been exported to **`ml/output/figures/`** and the written report to **`ml/output/eda_report.md`**.